# 🏗️ Scenario 3 — Detailed 3D Environment from a 2D Floor Plan  ·  Kaggle edition

**What this does:** take a flat 2D floor-plan image and draft a **detailed, textured 3D environment**
exported as a single **GLB** (plus GLTF + JSON metadata + a web viewer that actually loads).

**How the detail is made**
1. Parse the plan → room polygons + a wall network (door gaps stay as real openings).
2. Extrude real geometry: floor slabs per room, walls to height, and **baseboards** around every room.
3. Apply **procedural textures** (wood, carpet, tile, plaster, fabric) with planar UVs — no external assets.
4. Place **multi-part furniture** (a sofa is a base + back + arms + cushions + legs, not a box) with a
   deterministic settle + non-overlap solver.
5. Export one **GLB** with correct PBR materials/UVs + a JSON manifest, and a three.js viewer.

**Input:** drop your own floor-plan image into a Kaggle Dataset (it appears under `/kaggle/input/...`).
No input? It builds a synthetic apartment plan so the whole notebook runs end-to-end.
**Output:** everything is written to `/kaggle/working/outputs/` (Kaggle collects `/kaggle/working` automatically).

> Add your own textures at the marked step to push realism further; the defaults are procedurally generated.


## 📦 1 · Setup

In [ ]:
# Kaggle already ships numpy / opencv / pillow / matplotlib / pandas. Add trimesh + shapely.
import importlib, subprocess, sys
def _ensure(pairs):
    for mod, pkg in pairs:
        try:
            importlib.import_module(mod)
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
_ensure([("trimesh", "trimesh"), ("shapely", "shapely"),
         ("cv2", "opencv-python-headless"), ("PIL", "pillow"),
         ("mapbox_earcut", "mapbox-earcut")])   # trimesh needs an earcut engine

import os, json, math, zipfile
from pathlib import Path
import numpy as np
import cv2
import trimesh
from trimesh.creation import extrude_polygon
from trimesh.visual.texture import TextureVisuals
from trimesh.visual.material import PBRMaterial
from shapely.geometry import Polygon
from shapely.ops import unary_union
import shapely
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd
print("trimesh", trimesh.__version__, "| shapely", shapely.__version__,
      "| cv2", cv2.__version__, "| numpy", np.__version__)


## 📥 2 · Find (or build) a floor plan

In [ ]:
WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = WORKDIR / "outputs"
for _d in ["input", "3d", "textures", "visualizations", "web"]:
    (OUT / _d).mkdir(parents=True, exist_ok=True)
IN_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def generate_sample_plan(path):
    '''A clean synthetic apartment: white paper, black walls, a few door gaps.'''
    W, H, t = 1000, 720, 14
    img = np.full((H, W), 255, np.uint8)
    cv2.rectangle(img, (40, 40), (W - 40, H - 40), 0, t)      # outer shell
    cv2.line(img, (40, 360), (W - 40, 360), 0, t)             # horizontal divider
    cv2.line(img, (470, 40), (470, 360), 0, t)                # vertical (upper)
    cv2.line(img, (470, 360), (470, H - 40), 0, t)            # vertical (lower)
    cv2.line(img, (700, 360), (700, H - 40), 0, t)            # bathroom wall
    cv2.rectangle(img, (230, 352), (330, 368), 255, -1)       # door: living <-> kitchen
    cv2.rectangle(img, (462, 180), (478, 280), 255, -1)       # door: living <-> bedroom
    cv2.rectangle(img, (462, 500), (478, 560), 255, -1)       # door: kitchen <-> bath
    cv2.imwrite(str(path), img)
    return path

input_files = []
kag = Path("/kaggle/input")
if kag.exists():
    input_files = sorted(p for p in kag.rglob("*") if p.suffix.lower() in IN_EXT)
if not input_files:
    p = OUT / "input" / "sample_apartment.png"
    generate_sample_plan(p)
    input_files = [p]
    print("No floor plan under /kaggle/input -> using the built-in sample plan.")
print("Floor plans to process:")
for p in input_files:
    print("  -", p)


## 🔍 3 · Parse the plan into rooms + a wall network

In [ ]:
MIN_ROOM_PX = 2500          # ignore specks smaller than this
INK_K = 3                   # tiny close to tidy the scanned ink

def _interior_rooms(walls, H, W, k):
    '''Close door-sized gaps (kernel k) so rooms separate, then take interior free components.'''
    k = max(3, int(k) | 1)
    sep = cv2.morphologyEx(walls, cv2.MORPH_CLOSE, np.ones((k, k), np.uint8))
    free = cv2.bitwise_not(sep)
    ff = free.copy()
    cv2.floodFill(ff, np.zeros((H + 2, W + 2), np.uint8), (0, 0), 0)   # drop the outside
    n, labels, stats, cents = cv2.connectedComponentsWithStats(ff, connectivity=4)
    rooms = []
    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < MIN_ROOM_PX:
            continue
        comp = (labels == i).astype(np.uint8) * 255
        cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        c = max(cnts, key=cv2.contourArea)
        poly = cv2.approxPolyDP(c, 0.012 * cv2.arcLength(c, True), True).reshape(-1, 2)
        x, y, w, h = cv2.boundingRect(c)
        rooms.append({"poly_px": poly.astype(float), "bbox_px": [int(x), int(y), int(w), int(h)],
                      "area_px": area, "centroid_px": [float(cents[i][0]), float(cents[i][1])]})
    return rooms

def segment_rooms(walls, H, W):
    '''Sweep the door-closing kernel; keep the smallest k that reaches the max room count.

    Door gaps make every room one connected free region; closing a door-sized kernel
    separates them, but too large a kernel starts eroding rooms. The true room count is
    the plateau maximum.'''
    hi = max(15, min(W, H) // 3)
    ks = sorted(set(list(range(9, hi, 15)) + [hi]))
    results = [(k, _interior_rooms(walls, H, W, k)) for k in ks]
    maxc = max(len(r) for _k, r in results)
    for k, rooms in results:
        if len(rooms) == maxc:
            return rooms, k
    return results[-1][1], results[-1][0]

def parse_floor_plan(path, meters_per_px=None):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(path)
    H, W = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, walls = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    walls = cv2.morphologyEx(walls, cv2.MORPH_CLOSE, np.ones((INK_K, INK_K), np.uint8))
    ink = int(np.count_nonzero(walls)) / float(W * H)
    if ink > 0.5:                                  # light background / dark lines? invert
        walls = cv2.bitwise_not(walls)
    rooms, k = segment_rooms(walls, H, W)
    rooms.sort(key=lambda r: -r["area_px"])
    for idx, r in enumerate(rooms):
        r["id"] = idx
    if meters_per_px is None:
        meters_per_px = 14.0 / max(W, H)           # assume the long side is ~14 m
    print(f"{Path(path).name}: {W}x{H}px -> {len(rooms)} rooms "
          f"(door-close k={k}) @ {meters_per_px*100:.2f} cm/px")
    return {"path": str(path), "W": W, "H": H, "mpp": meters_per_px,
            "rooms": rooms, "walls_mask": walls}

plans = [parse_floor_plan(p) for p in input_files]


## 🎨 4 · Procedural textures (no external assets)

In [ ]:
# ---------------------------------------------------------------- textures
def _save(name, arr):
    img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    img.save(OUT / "textures" / f"{name}.png")
    return img

def tex_wood(size=512, seed=3):
    rng = np.random.default_rng(seed)
    arr = np.zeros((size, size, 3), float)
    row = size // 6
    base = np.array([158, 112, 66], float)
    for i in range(6):
        col = base * (1.0 + (rng.random() - 0.5) * 0.28)
        arr[i*row:(i+1)*row] = col
        arr[i*row:i*row+2] = col * 0.45                 # seam
        g = np.sin(np.linspace(0, rng.uniform(4, 9) * math.pi, size))[None, :, None]
        arr[i*row:(i+1)*row] += (g * 10.0)
    arr += rng.normal(0, 5, arr.shape)
    return _save("wood", arr)

def tex_carpet(size=512, seed=7):
    rng = np.random.default_rng(seed)
    base = np.array([96, 92, 104], float)
    arr = np.tile(base, (size, size, 1))
    arr += rng.normal(0, 16, arr.shape)
    return _save("carpet", arr)

def tex_tiles(size=512, n=4, seed=11):
    rng = np.random.default_rng(seed)
    step, g = size // n, max(4, size // 90)
    base = np.array([214, 218, 222], float)
    arr = np.tile(base, (size, size, 1))
    for i in range(n):
        for j in range(n):
            arr[i*step+g:(i+1)*step, j*step+g:(j+1)*step] *= (1.0 + (rng.random() - 0.5) * 0.07)
    grout = np.array([150, 154, 160], float)
    for k in range(0, size, step):
        arr[k:k+g, :] = grout
        arr[:, k:k+g] = grout
    arr += rng.normal(0, 3, arr.shape)
    return _save("tile", arr)

def tex_plaster(size=512, seed=5):
    rng = np.random.default_rng(seed)
    base = np.array([232, 229, 223], float)
    arr = np.tile(base, (size, size, 1))
    arr += rng.normal(0, 4.0, (size, size, 1))
    # soften with a cheap box blur
    k = 3
    pad = np.pad(arr, ((k, k), (k, k), (0, 0)), mode="edge")
    acc = np.zeros_like(arr)
    for dx in range(-k, k + 1):
        for dy in range(-k, k + 1):
            acc += pad[k+dx:k+dx+size, k+dy:k+dy+size]
    arr = acc / ((2*k+1) ** 2)
    return _save("plaster", arr)

def tex_fabric(size=256, seed=13):
    rng = np.random.default_rng(seed)
    base = np.array([74, 78, 96], float)
    arr = np.tile(base, (size, size, 1))
    weave = (np.sin(np.arange(size) / 2.0)[:, None] + np.sin(np.arange(size) / 2.0)[None, :])
    arr += weave[:, :, None] * 6.0
    arr += rng.normal(0, 7, arr.shape)
    return _save("fabric", arr)

TEX = {"wood": tex_wood(), "carpet": tex_carpet(), "tile": tex_tiles(),
       "plaster": tex_plaster(), "fabric": tex_fabric()}
# world size (metres) that one texture tile covers, per material
TEX_WORLD = {"wood": 2.0, "carpet": 1.6, "tile": 1.0, "plaster": 2.4, "fabric": 0.6}

def build_materials():
    mats = {}
    rough = {"wood": 0.72, "carpet": 1.0, "tile": 0.35, "plaster": 0.92, "fabric": 0.95}
    for k, img in TEX.items():
        mats[k] = PBRMaterial(name=k, baseColorTexture=img,
                              baseColorFactor=[1.0, 1.0, 1.0, 1.0],
                              metallicFactor=0.0, roughnessFactor=rough[k])
    flat = {
        "baseboard": ([0.96, 0.95, 0.92], 0.6, 0.0),
        "wood_dark": ([0.30, 0.20, 0.12], 0.7, 0.0),
        "wood_mid":  ([0.54, 0.37, 0.22], 0.7, 0.0),
        "metal":     ([0.72, 0.73, 0.76], 0.35, 0.9),
        "white":     ([0.93, 0.94, 0.95], 0.45, 0.0),
        "counter":   ([0.20, 0.22, 0.25], 0.30, 0.1),
        "glass":     ([0.74, 0.86, 0.90], 0.10, 0.0),
        "fabric_lt": ([0.62, 0.64, 0.74], 0.95, 0.0),
        "rug":       ([0.52, 0.28, 0.30], 0.98, 0.0),
    }
    for k, (rgb, r, m) in flat.items():
        mats[k] = PBRMaterial(name=k, baseColorFactor=[rgb[0], rgb[1], rgb[2], 1.0],
                              roughnessFactor=r, metallicFactor=m)
    return mats

MATS = build_materials()

def apply_flat(mesh, name):
    mesh.visual = TextureVisuals(material=MATS[name])

def apply_tex(mesh, name, axis="xy"):
    v = mesh.vertices
    if axis == "xy":
        u, w = v[:, 0], v[:, 1]
    else:                                   # walls: wrap horizontally along the wall, up the height
        u, w = v[:, 0] + v[:, 1], v[:, 2]
    s = TEX_WORLD[name]
    mesh.visual = TextureVisuals(uv=np.column_stack([u / s, w / s]), material=MATS[name])

print("textures:", ", ".join(f"{k} {im.size[0]}px" for k, im in TEX.items()))
print("materials:", len(MATS))


## 🏗️ 5 · Extrude detailed geometry (floors, walls, baseboards)

In [ ]:
WALL_H = 2.8        # metres
FLOOR_T = 0.12      # floor slab thickness
BASE_H = 0.12       # baseboard height
BASE_W = 0.05       # baseboard thickness

ROOM_FLOOR = {"living_room": "wood", "bedroom": "carpet", "office": "wood",
              "kitchen": "tile", "bathroom": "tile"}

def poly_m(poly_px, H, mpp):
    pts = [(x * mpp, (H - y) * mpp) for x, y in poly_px]     # flip Y so the plan isn't mirrored
    p = Polygon(pts).buffer(0)
    return p if (not p.is_empty and p.area > 1e-6) else None

def ring(outer, width):
    inner = outer.buffer(-width)
    if inner.is_empty:
        return None
    r = outer.difference(inner)
    return r if (not r.is_empty and r.area > 1e-9) else None

def room_type(idx, total):
    if idx == 0:
        return "living_room"
    if total > 4 and idx == 1:
        return "bedroom"
    if total > 3 and idx == total - 1:
        return "kitchen"
    return ["bedroom", "office", "bathroom"][idx % 3]

def build_room_geometry(plan):
    H, mpp = plan["H"], plan["mpp"]
    rooms_m, meshes = [], []

    # --- floors (per room) + baseboards (per room) ---
    for idx, r in enumerate(plan["rooms"]):
        p = poly_m(r["poly_px"], H, mpp)
        if p is None:
            continue
        t = room_type(idx, len(plan["rooms"]))
        rooms_m.append({"id": r["id"], "type": t, "poly": p, "area": p.area, "bounds": list(p.bounds)})

        floor = extrude_polygon(p, FLOOR_T)
        floor.apply_translation([0, 0, -FLOOR_T])
        apply_tex(floor, ROOM_FLOOR[t], axis="xy")
        meshes.append(("floor", r["id"], floor))

        b = ring(p, BASE_W)
        if b is not None:
            bb = extrude_polygon(b, BASE_H)
            apply_flat(bb, "baseboard")
            meshes.append(("baseboard", r["id"], bb))

        if t in ("living_room", "bedroom"):        # an inset rug under the furniture
            rug = p.buffer(-0.8)
            if (not rug.is_empty) and rug.area > 1.0:
                rm = extrude_polygon(rug, 0.02)
                rm.apply_translation([0, 0, 0.004])
                apply_flat(rm, "rug")
                meshes.append(("rug", r["id"], rm))

        minx, miny, maxx, maxy = p.bounds         # a ceiling light fixture
        cl = trimesh.creation.cylinder(radius=0.16, height=0.06, sections=24)
        cl.apply_translation([(minx + maxx) / 2, (miny + maxy) / 2, WALL_H - 0.05])
        apply_flat(cl, "white")
        meshes.append(("light", r["id"], cl))

    # --- walls: extrude the wall-mask contours, keeping room interiors as real holes ---
    cnts, hier = cv2.findContours(plan["walls_mask"], cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    hier = hier[0] if hier is not None else []
    n_wall = 0
    for i, c in enumerate(cnts):
        if cv2.contourArea(c) < (mpp ** 2) * 0.02:
            continue                                    # skip specks
        shell = poly_m(c.reshape(-1, 2), H, mpp)
        if shell is None:
            continue
        holes = []
        child = hier[i][2] if i < len(hier) else -1
        while child != -1:
            hp = poly_m(cnts[child].reshape(-1, 2), H, mpp)
            if hp is not None and hp.area > 1e-4:
                holes.append(list(hp.exterior.coords))
            child = hier[child][0]
        try:
            poly = Polygon(list(shell.exterior.coords), holes).buffer(0)
        except Exception:
            poly = shell
        if poly.is_empty or poly.area <= 1e-6:
            poly = shell
        try:
            wall = extrude_polygon(poly, WALL_H)
        except Exception:
            continue
        apply_tex(wall, "plaster", axis="xz")
        meshes.append(("wall", None, wall))
        n_wall += 1

    print(f"  {len(rooms_m)} rooms, {n_wall} wall pieces, "
          f"{sum(1 for k,_,_ in meshes if k=='baseboard')} baseboards")
    return rooms_m, meshes

scene_builds = []
for plan in plans:
    rooms_m, meshes = build_room_geometry(plan)
    scene_builds.append({"plan": plan, "rooms": rooms_m, "meshes": meshes})


## 🪑 6 · Multi-part furniture + settle/overlap solver

In [ ]:
def box_at(cx, cy, cz, w, d, h):
    m = trimesh.creation.box(extents=[w, d, h])
    m.apply_translation([cx, cy, cz])
    return m

# ---- part builders: author each item around (0,0) with the floor at z=0, +Y forward ----
def f_sofa(w, d, h):
    sh = h * 0.42
    p = [(box_at(0, 0, sh/2, w, d, sh), "fabric"),
         (box_at(0, d/2 - 0.11, sh + (h - sh)/2, w, 0.22, h - sh), "fabric"),
         (box_at(-w/2 + 0.10, 0, sh + 0.18, 0.20, d, 0.36), "fabric"),
         (box_at( w/2 - 0.10, 0, sh + 0.18, 0.20, d, 0.36), "fabric"),
         (box_at(0, -d*0.10, sh + 0.07, w - 0.5, d*0.55, 0.14), "fabric_lt")]
    for sx in (-1, 1):
        for sy in (-1, 1):
            p.append((box_at(sx*(w/2 - 0.10), sy*(d/2 - 0.10), 0.06, 0.10, 0.10, 0.12), "wood_dark"))
    return p

def f_armchair(w, d, h):
    return f_sofa(w, d, h)

def f_coffee_table(w, d, h):
    p = [(box_at(0, 0, h - 0.04, w, d, 0.06), "wood_mid")]
    for sx in (-1, 1):
        for sy in (-1, 1):
            p.append((box_at(sx*(w/2 - 0.06), sy*(d/2 - 0.06), (h - 0.04)/2, 0.07, 0.07, h - 0.04), "wood_dark"))
    return p

def f_tv_stand(w, d, h):
    p = [(box_at(0, 0, h/2, w, d, h), "wood_dark"),
         (box_at(0, 0, h + 0.35, w*0.9, 0.05, 0.7), "metal"),
         (box_at(0, 0.03, h + 0.35, w*0.82, 0.02, 0.6), "counter")]
    return p

def f_bed(w, d, h):
    p = [(box_at(0, 0, 0.16, w, d, 0.32), "wood_mid"),
         (box_at(0, 0.05, 0.40, w - 0.1, d - 0.1, 0.24), "white"),
         (box_at(0, d/2 - 0.02, 0.62, w, 0.06, 0.7), "wood_dark"),
         (box_at(-w*0.22, d*0.28, 0.54, w*0.4, 0.32, 0.14), "white"),
         (box_at( w*0.22, d*0.28, 0.54, w*0.4, 0.32, 0.14), "white"),
         (box_at(0, -d/2 + 0.15, 0.50, w - 0.15, 0.5, 0.08), "rug")]
    return p

def f_nightstand(w, d, h):
    return [(box_at(0, 0, h/2 + 0.08, w, d, h - 0.08), "wood_mid"),
            (box_at(0, -d/2 + 0.01, h*0.62, w - 0.06, 0.02, h*0.34), "wood_dark")] + \
           [(box_at(sx*(w/2 - 0.05), sy*(d/2 - 0.05), 0.04, 0.06, 0.06, 0.08), "wood_dark")
            for sx in (-1, 1) for sy in (-1, 1)]

def f_dresser(w, d, h):
    p = [(box_at(0, 0, h/2, w, d, h), "wood_mid")]
    for i in range(3):
        zc = h * (0.2 + i * 0.28)
        p.append((box_at(0, -d/2 - 0.005, zc, w - 0.08, 0.02, h * 0.2), "wood_dark"))
        p.append((box_at(0, -d/2 - 0.02, zc, w * 0.3, 0.03, 0.03), "metal"))
    return p

def f_wardrobe(w, d, h):
    p = [(box_at(0, 0, h/2, w, d, h), "wood_mid"),
         (box_at(0, -d/2 - 0.005, h*0.5, w/2 - 0.02, 0.02, h - 0.1), "wood_dark"),
         (box_at(0, -d/2 - 0.02, h*0.5, 0.03, 0.03, 0.18), "metal")]
    p += [(box_at(sx*(w/2 - 0.04), sy*(d/2 - 0.04), 0.06, 0.08, 0.08, 0.12), "wood_dark")
          for sx in (-1, 1) for sy in (-1, 1)]
    return p

def f_counter(w, d, h):
    return [(box_at(0, 0, (h - 0.04)/2, w, d, h - 0.04), "wood_mid"),
            (box_at(0, 0, h - 0.02, w + 0.03, d + 0.03, 0.05), "counter"),
            (box_at(-w*0.25, -d/2 - 0.005, h*0.5, w*0.3, 0.02, h*0.5), "wood_dark")]

def f_fridge(w, d, h):
    return [(box_at(0, 0, h/2, w, d, h), "white"),
            (box_at(0, -d/2 - 0.01, h*0.62, w - 0.06, 0.02, h*0.72), "metal"),
            (box_at(0, -d/2 - 0.03, h*0.75, 0.03, 0.05, 0.10), "metal")]

def f_table(w, d, h):
    p = [(box_at(0, 0, h, w, d, 0.05), "wood_mid")]
    p += [(box_at(sx*(w/2 - 0.07), sy*(d/2 - 0.07), (h - 0.05)/2, 0.07, 0.07, h - 0.05), "wood_dark")
          for sx in (-1, 1) for sy in (-1, 1)]
    return p

def f_tub(w, d, h):
    p = [(box_at(0, 0, h/2, w, d, h), "white")]
    inner = box_at(0, 0, h - 0.10, w - 0.16, d - 0.16, 0.20)
    p.append((inner, "counter"))     # dark recess reads as the basin
    return p

def f_sink_unit(w, d, h):
    return [(box_at(0, 0, (h - 0.04)/2, w, d, h - 0.04), "white"),
            (box_at(0, 0, h - 0.02, w + 0.03, d + 0.03, 0.05), "counter"),
            (box_at(0, 0, h + 0.02, w * 0.6, d * 0.6, 0.06), "white")]

def f_toilet(w, d, h):
    return [(box_at(0, -d/2 + 0.12, h*0.35, w, d*0.45, h*0.7), "white"),
            (box_at(0, d*0.15, 0.20, w*0.9, d*0.5, 0.4), "white"),
            (box_at(0, d*0.15, 0.42, w*0.75, d*0.42, 0.05), "white")]

def f_desk(w, d, h):
    return [(box_at(0, 0, h, w, d, 0.05), "wood_mid"),
            (box_at(0, d/2 - 0.03, h/2, w - 0.1, 0.04, h - 0.05), "wood_mid")] + \
           [(box_at(sx*(w/2 - 0.06), sy*(d/2 - 0.06), (h - 0.05)/2, 0.06, 0.06, h - 0.05), "metal")
            for sx in (-1, 1) for sy in (-1, 1)]

def f_chair(w, d, h):
    return [(box_at(0, 0, h*0.5, w, d, 0.06), "fabric_lt"),
            (box_at(0, d/2 - 0.04, h*0.5 + 0.22, w, 0.05, 0.44), "fabric_lt")] + \
           [(box_at(sx*(w/2 - 0.04), sy*(d/2 - 0.04), h*0.22, 0.05, 0.05, h*0.44), "metal")
            for sx in (-1, 1) for sy in (-1, 1)]

def f_shelf(w, d, h):
    p = [(box_at(-w/2 + 0.02, 0, h/2, 0.04, d, h), "wood_mid"),
         (box_at( w/2 - 0.02, 0, h/2, 0.04, d, h), "wood_mid"),
         (box_at(0, d/2 - 0.02, h/2, w, 0.03, h), "wood_dark")]
    for i in range(4):
        p.append((box_at(0, 0, h * (0.08 + i * 0.28), w, d, 0.03), "wood_mid"))
    return p

FURN = {
    "living_room": [("Sofa", (2.10, 0.85, 0.80), f_sofa), ("CoffeeTable", (1.10, 0.60, 0.45), f_coffee_table),
                    ("TvStand", (1.60, 0.45, 0.50), f_tv_stand), ("Armchair", (0.85, 0.85, 0.80), f_armchair)],
    "bedroom": [("Bed", (2.00, 1.60, 0.70), f_bed), ("Nightstand", (0.50, 0.45, 0.55), f_nightstand),
                ("Dresser", (1.20, 0.50, 0.90), f_dresser), ("Wardrobe", (1.20, 0.60, 2.00), f_wardrobe)],
    "kitchen": [("Counter", (2.40, 0.60, 0.90), f_counter), ("Counter2", (1.60, 0.60, 0.90), f_counter),
                ("Fridge", (0.70, 0.70, 1.80), f_fridge), ("Table", (1.10, 0.75, 0.75), f_table)],
    "bathroom": [("Tub", (1.70, 0.75, 0.55), f_tub), ("SinkUnit", (0.60, 0.50, 0.85), f_sink_unit),
                 ("Toilet", (0.40, 0.60, 0.75), f_toilet)],
    "office": [("Desk", (1.60, 0.80, 0.75), f_desk), ("Chair", (0.55, 0.55, 0.90), f_chair),
               ("Shelf", (1.00, 0.40, 1.90), f_shelf)],
}
MARGIN = 0.25

def place_furniture(rooms_m):
    items = []
    for r in rooms_m:
        minx, miny, maxx, maxy = r["bounds"]
        tpl = FURN.get(r["type"], FURN["bedroom"])
        cx, cy, row_h = minx + MARGIN, maxy - MARGIN, 0.0
        for name, (w, d, h), _fn in tpl:
            if cx + w > maxx - MARGIN:
                cx, cy, row_h = minx + MARGIN, cy - row_h - MARGIN, 0.0
            if cy - d < miny:
                break
            items.append({"type": name, "room_id": r["id"], "size": [w, d, h],
                          "position": [cx, cy - d]})
            cx += w + MARGIN
            row_h = max(row_h, d)
    return items

def solve_placement(items, rooms_m, iters=80):
    by_id = {r["id"]: r for r in rooms_m}
    for _ in range(iters):
        moved = False
        for a in items:                                   # keep inside the room bounds
            r = by_id[a["room_id"]]
            minx, miny, maxx, maxy = r["bounds"]
            w, d, _ = a["size"]
            nx = min(max(a["position"][0], minx + 0.02), maxx - w - 0.02)
            ny = min(max(a["position"][1], miny + 0.02), maxy - d - 0.02)
            if abs(nx - a["position"][0]) > 1e-6 or abs(ny - a["position"][1]) > 1e-6:
                a["position"][0], a["position"][1] = nx, ny
                moved = True
        for i in range(len(items)):                       # separate overlaps within a room
            for j in range(i + 1, len(items)):
                a, b = items[i], items[j]
                if a["room_id"] != b["room_id"]:
                    continue
                ax, ay = a["position"]; bx, by = b["position"]
                aw, ad = a["size"][0], a["size"][1]
                bw, bd = b["size"][0], b["size"][1]
                ox = min(ax + aw, bx + bw) - max(ax, bx)
                oy = min(ay + ad, by + bd) - max(ay, by)
                if ox > 0 and oy > 0:
                    if ox < oy:
                        s = -1 if ax < bx else 1
                        a["position"][0] += s * (ox/2 + 0.01); b["position"][0] -= s * (ox/2 + 0.01)
                    else:
                        s = -1 if ay < by else 1
                        a["position"][1] += s * (oy/2 + 0.01); b["position"][1] -= s * (oy/2 + 0.01)
                    moved = True
        if not moved:
            break
    return items

def furniture_meshes(items):
    out = []
    for it in items:
        w, d, h = it["size"]
        cx, cy = it["position"][0] + w/2, it["position"][1] + d/2
        fn = it["_fn"]
        for mesh, matname in fn(w, d, h):
            mesh.apply_translation([cx, cy, 0.0])
            if matname in TEX:
                apply_tex(mesh, matname, axis="xy")
            else:
                apply_flat(mesh, matname)
            out.append((it, mesh))
    return out

for sb in scene_builds:
    items = solve_placement(place_furniture(sb["rooms"]), sb["rooms"])
    tmap = {r["id"]: r["type"] for r in sb["rooms"]}
    fbn = {name: fn for t in FURN.values() for name, _s, fn in t}
    for it in items:
        it["_room_type"] = tmap[it["room_id"]]
        it["_fn"] = fbn[it["type"]]
    parts = furniture_meshes(items)
    sb["furniture"] = items
    sb["meshes"] += [("furniture", it["room_id"], m) for it, m in parts]
    print(f"  scene: {len(sb['rooms'])} rooms, {len(items)} furniture items, "
          f"{len(parts)} furniture parts")


## 🎮 7 · Export GLB / GLTF + JSON manifest

In [ ]:
def export_scene(sb, scene_id):
    scene = trimesh.Scene()
    for kind, room_id, m in sb["meshes"]:
        base = f"{kind}_{room_id}" if room_id is not None else kind
        scene.add_geometry(m, node_name=f"{base}_{len(scene.geometry)}")
    glb = OUT / "3d" / f"scene_{scene_id}.glb"
    scene.export(str(glb))
    try:
        gdir = OUT / "3d" / "gltf"               # keep the multi-buffer glTF tidy in a subdir
        gdir.mkdir(parents=True, exist_ok=True)
        scene.export(str(gdir / f"scene_{scene_id}.gltf"))
    except Exception as e:
        print("  (gltf export skipped:", e, ")")
    bounds = scene.bounds.tolist() if scene.bounds is not None else [[0, 0, 0], [0, 0, 0]]
    meta = {
        "scene_id": scene_id,
        "source": Path(sb["plan"]["path"]).name,
        "scale_m_per_px": sb["plan"]["mpp"],
        "bounds_m": bounds,
        "textures": sorted(TEX.keys()),
        "detail": {"wall_height_m": WALL_H, "floor_thickness_m": FLOOR_T,
                   "baseboards": True, "multi_part_furniture": True},
        "rooms": [{"id": r["id"], "type": r["type"], "area_m2": round(r["area"], 2),
                   "bounds": [round(x, 3) for x in r["bounds"]]} for r in sb["rooms"]],
        "furniture": [{"type": it["type"], "room_id": it["room_id"], "size": it["size"],
                       "position": [round(v, 3) for v in it["position"]]} for it in sb["furniture"]],
        "interaction_points": [{"id": f"room_{r['id']}", "type": "room",
                                "position": [round((r["bounds"][0] + r["bounds"][2]) / 2, 3), 0,
                                             round((r["bounds"][1] + r["bounds"][3]) / 2, 3)],
                                "actions": ["enter", "teleport"]} for r in sb["rooms"]],
        "lighting": {"ambient": 0.6, "directional": 1.4, "shadows": True},
        "physics": {"gravity": [0, 0, -9.81], "collision": "aabb-separation",
                    "solver": "settle-and-separate", "iterations": 80},
    }
    (OUT / "3d" / f"scene_{scene_id}.json").write_text(json.dumps(meta, indent=2))
    return glb, meta

exports = [export_scene(sb, i + 1) for i, sb in enumerate(scene_builds)]
for glb, meta in exports:
    print(f"  exported {glb.name}  ({glb.stat().st_size/1024:.0f} KB)  "
          f"{len(meta['rooms'])} rooms, {len(meta['furniture'])} items")


## 🌐 8 · Web viewer (three.js ESM + GLTFLoader + OrbitControls)

In [ ]:
VIEWER = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>__TITLE__</title>
<style>
  html,body{margin:0;height:100%;overflow:hidden;background:#0e1116;font-family:system-ui,Arial,sans-serif}
  #hud{position:fixed;top:12px;left:12px;z-index:2;color:#e8eef7;background:rgba(14,17,22,.72);
       padding:10px 14px;border-radius:10px;font-size:13px;line-height:1.5}
  #hud b{color:#7fd1ff}
</style>
</head>
<body>
<div id="hud"><b>__TITLE__</b><br/>rooms: __ROOMS__ &nbsp;|&nbsp; objects: __ITEMS__<br/>
drag: orbit &nbsp;·&nbsp; wheel: zoom &nbsp;·&nbsp; right-drag: pan</div>
<script type="importmap">
{ "imports": {
  "three": "https://unpkg.com/three@0.160.0/build/three.module.js",
  "three/addons/": "https://unpkg.com/three@0.160.0/examples/jsm/"
} }
</script>
<script type="module">
import * as THREE from 'three';
import { OrbitControls } from 'three/addons/controls/OrbitControls.js';
import { GLTFLoader } from 'three/addons/loaders/GLTFLoader.js';

const scene = new THREE.Scene();
scene.background = new THREE.Color(0x0e1116);
const camera = new THREE.PerspectiveCamera(55, innerWidth/innerHeight, 0.05, 500);
camera.position.set(10, 12, 14);
const renderer = new THREE.WebGLRenderer({ antialias:true });
renderer.setSize(innerWidth, innerHeight);
renderer.setPixelRatio(Math.min(devicePixelRatio, 2));
renderer.shadowMap.enabled = true;
renderer.shadowMap.type = THREE.PCFSoftShadowMap;
renderer.outputColorSpace = THREE.SRGBColorSpace;
document.body.appendChild(renderer.domElement);

scene.add(new THREE.HemisphereLight(0xdfe8ff, 0x30343c, 0.85));
const sun = new THREE.DirectionalLight(0xffffff, 2.0);
sun.position.set(12, 22, 8);
sun.castShadow = true;
sun.shadow.mapSize.set(2048, 2048);
sun.shadow.bias = -0.0005;
scene.add(sun);

const controls = new OrbitControls(camera, renderer.domElement);
controls.enableDamping = true;

new GLTFLoader().load('__GLB__', (gltf) => {
  scene.add(gltf.scene);
  const box = new THREE.Box3().setFromObject(gltf.scene);
  const size = box.getSize(new THREE.Vector3());
  const center = box.getCenter(new THREE.Vector3());
  controls.target.copy(center);
  const r = Math.max(size.x, size.y, size.z);
  camera.position.copy(center).add(new THREE.Vector3(r, r*1.1, r));
  camera.near = r/100; camera.far = r*50; camera.updateProjectionMatrix();
  controls.update();
}, undefined, (err) => { console.error(err); });

addEventListener('resize', () => {
  camera.aspect = innerWidth/innerHeight; camera.updateProjectionMatrix();
  renderer.setSize(innerWidth, innerHeight);
});
(function loop(){ requestAnimationFrame(loop); controls.update(); renderer.render(scene, camera); })();
</script>
</body>
</html>"""

for i, (glb, meta) in enumerate(exports, 1):
    html = (VIEWER.replace("__TITLE__", f"Environment {i}")
                  .replace("__GLB__", glb.name)
                  .replace("__ROOMS__", str(len(meta["rooms"])))
                  .replace("__ITEMS__", str(len(meta["furniture"]))))
    (OUT / "web" / f"viewer_{i}.html").write_text(html)
    print(f"  wrote viewer_{i}.html  ->  loads {glb.name}")


## 📊 9 · Visualize + summary

In [ ]:
def visualize(sb, scene_id):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    for r in sb["rooms"]:
        minx, miny, maxx, maxy = r["bounds"]
        ax1.add_patch(Rectangle((minx, miny), maxx - minx, maxy - miny,
                                ec="black", fc="#eaf2ff", alpha=0.9, lw=1.4))
        ax1.text((minx + maxx) / 2, (miny + maxy) / 2, r["type"],
                 ha="center", va="center", fontsize=7)
    for it in sb["furniture"]:
        x, y = it["position"][0], it["position"][1]
        w, d = it["size"][0], it["size"][1]
        ax1.add_patch(Rectangle((x, y), w, d, ec="none", fc="#d98b2b", alpha=0.85))
    ax1.set_aspect("equal"); ax1.set_title(f"Scene {scene_id}: plan (top view)")
    ax1.set_xlabel("x (m)"); ax1.set_ylabel("y (m)"); ax1.grid(alpha=0.25)

    ax2 = fig.add_subplot(122, projection="3d")
    shown = 0
    for kind, _rid, m in sb["meshes"]:
        if kind == "wall":
            continue
        v, f = m.vertices, m.faces
        try:
            ax2.plot_trisurf(v[:, 0], v[:, 1], v[:, 2], triangles=f,
                             shade=True, cmap="cividis", linewidth=0, alpha=0.95)
            shown += 1
        except Exception:
            pass
        if shown > 60:
            break
    ax2.set_title(f"Scene {scene_id}: 3D (floors + furniture)")
    ax2.set_box_aspect((1, 1, 0.4))
    ax2.set_xlabel("x"); ax2.set_ylabel("y"); ax2.set_zlabel("z")
    plt.tight_layout()
    p = OUT / "visualizations" / f"scene_{scene_id}.png"
    fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    return p

rows = []
for i, sb in enumerate(scene_builds, 1):
    visualize(sb, i)
    rows.append({"scene": i, "source": Path(sb["plan"]["path"]).name,
                 "rooms": len(sb["rooms"]), "furniture": len(sb["furniture"]),
                 "meshes": len(sb["meshes"])})
print("Summary:")
print(pd.DataFrame(rows).to_string(index=False))


## 💾 10 · Package results

In [ ]:
README = OUT / "README.md"
README.write_text(
    "# Detailed 3D Environment from a 2D Floor Plan\n\n"
    "Generated by `ar_vr_environment_kaggle.ipynb`.\n\n"
    "- `3d/scene_*.glb`   single-file binary glTF with PBR materials + embedded textures\n"
    "- `3d/gltf/`         JSON glTF + external buffers (interchange)\n"
    "- `3d/scene_*.json`  manifest: rooms, furniture, interaction points, lighting\n"
    "- `textures/*.png`   the procedural textures used\n"
    "- `visualizations/*.png`  plan + 3D previews\n"
    "- `web/viewer_*.html`  three.js viewer (serve next to the matching .glb)\n\n"
    "Detail includes: extruded walls with real door gaps, per-room floors, baseboards,\n"
    "rugs, ceiling lights, and multi-part furniture. Swap `textures/` for scanned\n"
    "materials, or add a depth/segmentation model in the parse step to go further.\n\n"
    "Targets: WebXR / three.js / Babylon, Unity, Unreal, ARKit, ARCore.\n"
)
zip_path = WORKDIR / "environments.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(OUT))
print("ZIP:", zip_path, f"({zip_path.stat().st_size/1024:.0f} KB)")
print("Files under /kaggle/working are saved as the notebook output automatically.")
print()
print("Done. Serve outputs/web/ next to outputs/3d/ and open viewer_1.html to inspect the environment in 3D.")
